# The Deutsch Algorithm

## The first quantum algorithm to show a provable speedup over its classical counterpart. Tiny by today's standards — one qubit's worth of advantage — but historically pivotal: it demonstrated that *quantum interference* can replace *evaluation count*.

# 1. The problem

## You are given a black-box (oracle) implementing a function

$$ \Large  f : \{0, 1\} \to \{0, 1\}. $$

## There are exactly four such functions, falling into two categories:

## - **Constant**: $f(0) = f(1)$ (two of these: always-0 and always-1).
## - **Balanced**: $f(0) \neq f(1)$ (the other two: identity and NOT).

## **Task**: decide whether $f$ is constant or balanced. **Classically**: you must query $f$ twice (compute $f(0)$ and $f(1)$). **Quantumly**: one query suffices.

# 2. The quantum oracle

## We need a reversible (unitary) version of $f$. The standard choice is

$$ \Large  U_f : |x\rangle|y\rangle \mapsto |x\rangle|y \oplus f(x)\rangle, $$

## where $\oplus$ is XOR. The first qubit (input) is preserved; the second qubit (output) is XOR-ed with $f(x)$. This is unitary for any function $f$ (it's a permutation of the four basis states).

# 3. The algorithm

## 1. Initialise two qubits as $|0\rangle|1\rangle$.
## 2. Apply $H \otimes H$. The state becomes $|+\rangle|-\rangle$.
## 3. Apply $U_f$.
## 4. Apply $H$ to the first qubit.
## 5. Measure the first qubit.

## Outcome 0 ⇒ $f$ is constant; outcome 1 ⇒ $f$ is balanced. **One query, deterministic answer.**

# 4. Why it works — the phase kickback trick

## When the second qubit is in state $|-\rangle = (|0\rangle - |1\rangle)/\sqrt 2$:

$$ \Large  U_f |x\rangle|{-}\rangle = |x\rangle \cdot \frac{1}{\sqrt 2}(|f(x)\rangle - |1 \oplus f(x)\rangle) = (-1)^{f(x)} |x\rangle |{-}\rangle. $$

## So the function value $f(x)$ has been transferred from the *target register* into the **phase** of the *control register*. This is **phase kickback** — it is the single most important trick in quantum algorithms.

## After step 2 the input register is in $|+\rangle = (|0\rangle + |1\rangle)/\sqrt 2$. After step 3 it becomes

$$ \Large  \tfrac{1}{\sqrt 2}\big((-1)^{f(0)}|0\rangle + (-1)^{f(1)}|1\rangle\big). $$

## If $f$ is constant, both signs match — the state is $\pm|+\rangle$ and the final Hadamard yields $|0\rangle$. If $f$ is balanced, the signs disagree — the state is $\pm|-\rangle$ and the final Hadamard yields $|1\rangle$.

In [1]:
# Run Deutsch's algorithm on all four functions using Qiskit
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

def deutsch_oracle(case):
    qc = QuantumCircuit(2, name=case)
    if case == 'const0':           pass
    elif case == 'const1':         qc.x(1)
    elif case == 'identity':       qc.cx(0, 1)
    elif case == 'not':            qc.cx(0, 1); qc.x(1)
    return qc

for case in ['const0', 'const1', 'identity', 'not']:
    qc = QuantumCircuit(2, 1)
    qc.x(1)                          # ancilla starts in |1>
    qc.h([0, 1])
    qc.compose(deutsch_oracle(case), inplace=True)
    qc.h(0)
    qc.measure(0, 0)
    sv = Statevector.from_label('00').evolve(qc.remove_final_measurements(inplace=False))
    print(f'{case:8s}: P(0)={sv.probabilities([0])[0]:.3f}, P(1)={sv.probabilities([0])[1]:.3f}')

const0  : P(0)=1.000, P(1)=0.000
const1  : P(0)=1.000, P(1)=0.000
identity: P(0)=0.000, P(1)=1.000
not     : P(0)=0.000, P(1)=1.000


# 5. The speedup

## Classical: 2 queries to be sure. Quantum: 1 query. A factor of 2, not earth-shattering — but the same idea (phase kickback + interference) scales up to **exponential** speedups in Deutsch–Jozsa, Bernstein–Vazirani, Simon, and Shor.

# Recap

## - Decide if $f: \{0,1\} \to \{0,1\}$ is constant or balanced.
## - Quantum: prepare $|+\rangle|-\rangle$, apply $U_f$, Hadamard, measure.
## - **Phase kickback** turns $f(x)$ into a relative phase $(-1)^{f(x)}$.
## - 1 query vs 2 classical — the warm-up for everything that follows.